In [ ]:
# import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import sqlite3

### **Load the Datasets**
In this project, we will be utilizing the:
- `im.db` SQL Database
- `bom.movie_gross.csv` CSV Dataset
for our analysis. 

The code below connects to the SQL database, and reads the contents in the database.

In [2]:
# connect to the SQLite database
conn = sqlite3.connect('data/im.db')

# view the tables in the database
q_read_data = """SELECT name 
                 FROM sqlite_master 
                 WHERE type='table';"""
pd.read_sql(q_read_data, conn)

,name
0,movie_basics
1,directors
2,known_for
3,movie_akas
4,movie_ratings
5,persons
6,principals
7,writers


The `im.db` database contains a number of tables, which include:
- movie_basics
- directors
- known_for
- movie_akas
- movie_ratings
- persons
- principals
- writers

In [5]:
# read data from movie_basics table
q_movie_basics = """SELECT *
                    FROM movie_basics;"""
movie_basics_df = pd.read_sql(q_movie_basics, conn)
movie_basics_df.head()

,movie_id,primary_title,original_title,start_year,runtime_minutes,genres
0,tt0063540,Sunghursh,Sunghursh,2013,175.0,"Action,Crime,Drama"
1,tt0066787,One Day Before the Rainy Season,Ashad Ka Ek Din,2019,114.0,"Biography,Drama"
2,tt0069049,The Other Side of the Wind,The Other Side of the Wind,2018,122.0,Drama
3,tt0069204,Sabse Bada Sukh,Sabse Bada Sukh,2018,NaN,"Comedy,Drama"
4,tt0100275,The Wandering Soap Opera,La Telenovela Errante,2017,80.0,"Comedy,Drama,Fantasy"


In [8]:
# check the info of the movie_basics_df
movie_basics_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 146144 entries, 0 to 146143
Data columns (total 6 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   movie_id         146144 non-null  object 
 1   primary_title    146144 non-null  object 
 2   original_title   146123 non-null  object 
 3   start_year       146144 non-null  int64  
 4   runtime_minutes  114405 non-null  float64
 5   genres           140736 non-null  object 
dtypes: float64(1), int64(1), object(4)
memory usage: 6.7+ MB


The `movie_basics` table contains a total of 146,144 entries, and a total of 6 columns, including **movie_id, primary_title, original_title, start_year, runtime_minutes, and genres**

In [7]:
# read from the movie_ratings table
q_movie_ratings = """SELECT *
                     FROM movie_ratings;"""
movie_ratings_df = pd.read_sql(q_movie_ratings, conn)
movie_ratings_df.head()

,movie_id,averagerating,numvotes
0,tt10356526,8.3,31
1,tt10384606,8.9,559
2,tt1042974,6.4,20
3,tt1043726,4.2,50352
4,tt1060240,6.5,21


In [10]:
# check the info of the dataframe
movie_ratings_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 73856 entries, 0 to 73855
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   movie_id       73856 non-null  object 
 1   averagerating  73856 non-null  float64
 2   numvotes       73856 non-null  int64  
dtypes: float64(1), int64(1), object(1)
memory usage: 1.7+ MB


The `movie_ratings` table contains a total of 73856 entries, with a total of 3 columns, including **movie_id, averagerating, and numvotes**.

In the database, we can combine the `movie_basics` and `movie_ratings` since they share a common column, which is **movie_id**

In [11]:
# join the movie_basics_df & movie_ratings_df dataframes using pandas
movie_df = pd.merge(movie_basics_df, movie_ratings_df, on='movie_id', how='inner')
movie_df.head()

,movie_id,primary_title,original_title,start_year,runtime_minutes,genres,averagerating,numvotes
0,tt0063540,Sunghursh,Sunghursh,2013,175.0,"Action,Crime,Drama",7.0,77
1,tt0066787,One Day Before the Rainy Season,Ashad Ka Ek Din,2019,114.0,"Biography,Drama",7.2,43
2,tt0069049,The Other Side of the Wind,The Other Side of the Wind,2018,122.0,Drama,6.9,4517
3,tt0069204,Sabse Bada Sukh,Sabse Bada Sukh,2018,NaN,"Comedy,Drama",6.1,13
4,tt0100275,The Wandering Soap Opera,La Telenovela Errante,2017,80.0,"Comedy,Drama,Fantasy",6.5,119


Next, we load the bom.movie_gross.csv dataset.

In [21]:
# load the movie_gross dataset
movie_gross_df = pd.read_csv('data/bom.movie_gross.csv')
movie_gross_df.head()

,title,studio,domestic_gross,foreign_gross,year
0,Toy Story 3,BV,415000000.0,652000000,2010
1,Alice in Wonderland (2010),BV,334200000.0,691300000,2010
2,Harry Potter and the Deathly Hallows Part 1,WB,296000000.0,664300000,2010
3,Inception,WB,292600000.0,535700000,2010
4,Shrek Forever After,P/DW,238700000.0,513900000,2010


In [22]:
# check the info of the movie_gross_df
movie_gross_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3387 entries, 0 to 3386
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   title           3387 non-null   object 
 1   studio          3382 non-null   object 
 2   domestic_gross  3359 non-null   float64
 3   foreign_gross   2037 non-null   object 
 4   year            3387 non-null   int64  
dtypes: float64(1), int64(1), object(3)
memory usage: 132.4+ KB


The `.info()` function provides a general summary of the dataframe, giving details like:
- Total number of **entries**(rows)
- The range of entries (0 to 3386)
- Total number of **columns**
- Total number of non-null values
- Data types for each column:
    - `object` -> for string
    - `int64` -> for integers
    - `float64` -> for floating point integers
This information aids in detecting missing data and determining whether columns require standardization.

The `movie_gross` dataframe has a total of `3387` records, and a total of `5` columns.

From the information, we can see that **foreign_gross** is displayed as an object data type. We can convert it to its correct data type, which is float.

In [26]:
# convert the foreign_gross column to numeric
movie_gross_df['foreign_gross'] = movie_gross_df['foreign_gross'].replace(",", "", regex=True).astype(float)

# display the first 5 rows
print(movie_gross_df['foreign_gross'].head())

0    652000000.0
1    691300000.0
2    664300000.0
3    535700000.0
4    513900000.0
Name: foreign_gross, dtype: float64


In [23]:
# summary statistics for movie_df
movie_df.describe()

,start_year,runtime_minutes,averagerating,numvotes
count,73856.000000,66236.000000,73856.000000,7.385600e+04
mean,2014.276132,94.654040,6.332729,3.523662e+03
std,2.614807,208.574111,1.474978,3.029402e+04
min,2010.000000,3.000000,1.000000,5.000000e+00
25%,2012.000000,81.000000,5.500000,1.400000e+01
50%,2014.000000,91.000000,6.500000,4.900000e+01
75%,2016.000000,104.000000,7.400000,2.820000e+02
max,2019.000000,51420.000000,10.000000,1.841066e+06


In [27]:
# summary statistics for movie_gross_df
movie_gross_df.describe()

,domestic_gross,foreign_gross,year
count,3.359000e+03,2.037000e+03,3387.000000
mean,2.874585e+07,7.487281e+07,2013.958075
std,6.698250e+07,1.374106e+08,2.478141
min,1.000000e+02,6.000000e+02,2010.000000
25%,1.200000e+05,3.700000e+06,2012.000000
50%,1.400000e+06,1.870000e+07,2014.000000
75%,2.790000e+07,7.490000e+07,2016.000000
max,9.367000e+08,9.605000e+08,2018.000000


The `.describe()` function provides summary statistics for numeric columns in the dataframes. It provides details such as:
- **count**: total number of nun-null records
- **mean**: Average value of each column
- **std (standard deviation)**: how spread out the values are
- **min**: Minimum value
- **25%**: 25th Percentile
- **50%**: median(mid value)
- **75%**: 75th percentile
- **max**: Maximum value 

From the `info()` function, we have seen there are missing values in both dataframes. We can check how many missing values are present in each column in the dataframes. 

In [35]:
# check for null values in movie_df
print(movie_df.isnull().sum())

movie_id              0
primary_title         0
original_title        0
start_year            0
runtime_minutes    7620
genres              804
averagerating         0
numvotes              0
dtype: int64


In [36]:
# check for null values in movie_gross_df
print(movie_gross_df.isnull().sum())

title                0
studio               5
domestic_gross      28
foreign_gross     1350
year                 0
dtype: int64


## **Data Cleaning**
### Movie_df Data Cleaning
In this dataframe, we will deal with the missing values by:
- Filling missing `runtime_minutes` values with the median
- Filling missing `genres` with the string **"Unknown"**

In [ ]:
# movie_df data cleaning
def clean_movie_df(df):
    # fill missing values in 'genres' with unknown
    movie_df['genres'].fillna('Unknown', inplace=True)

    # fill missing values in 'runtime_minutes' with the median
    runtime_median = movie_df['runtime_minutes'].median()
    movie_df['runtime_minutes'].fillna(runtime_median, inplace=True)
    
    return df

In [42]:
# call the function to clean the movie_df
clean_movie_df(movie_df)

# re-check for null values in movie_df
print("Null values after cleaning:")
print(movie_df.isnull().sum())

Null values after cleaning:
movie_id           0
primary_title      0
original_title     0
start_year         0
runtime_minutes    0
genres             0
averagerating      0
numvotes           0
dtype: int64


/tmp/ipykernel_388490/1821563200.py:8: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  movie_df['runtime_minutes'].fillna(runtime_median, inplace=True)


### Movie_gross Data Cleaning
In this dataframe, we will deal with missing values by:
- Drop the rows in `studio` and `domestic_gross` with missing values
- Filling the `foreign_gross` column with the median

In [43]:
# movie_gross_df data cleaning
def clean_movie_gross(df):

    # drop rown in 'studio' and 'domestic_gross'
    movie_gross_df.dropna(subset=['studio', 'domestic_gross'], inplace=True, axis=0)

    # fill the 'foreign_gross' column with median
    median_gross = movie_gross_df['foreign_gross'].median()
    movie_gross_df['foreign_gross'].fillna(median_gross, inplace=True)

    return df

In [44]:
# call the function to clean the movie_gross_df
clean_movie_gross(movie_gross_df)

# re-check for null values in movie_gross_df
print("Null values after cleaning:")
print(movie_gross_df.isnull().sum())

Null values after cleaning:
title             0
studio            0
domestic_gross    0
foreign_gross     0
year              0
dtype: int64


/tmp/ipykernel_388490/3079501367.py:9: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  movie_gross_df['foreign_gross'].fillna(median_gross, inplace=True)


### **Standardizing Column Names**
Next, we can standardize the columns in each dataframe to ensure readability and uniformity across dataframes. We can do this using the following methods:
- `str.strip()`: used to remove any trailing whitespace in a string
- `str.lower()`: used to convert a string to lowercase
- `str.replace()`: used to replace characters in a string

In [47]:
# standardize column names in movie_df
movie_df.columns = (
    movie_df.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_')
)
# replace 'averagerating' and 'numvotes'
movie_df = movie_df.rename(
    columns = {
        'averagerating': 'average_rating',
        'numvotes': 'number_of_votes'
    }
)

print("Standardized column names in movie_df: ", movie_df.columns.tolist())

# standardize column names in movie_gross_df
movie_gross_df.columns = (
    movie_gross_df.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_')
)
print("Standardized column names in movie_gross_df: ", movie_gross_df.columns.tolist())

Standardized column names in movie_df:  ['movie_id', 'primary_title', 'original_title', 'start_year', 'runtime_minutes', 'genres', 'average_rating', 'number_of_votes']
Standardized column names in movie_gross_df:  ['title', 'studio', 'domestic_gross', 'foreign_gross', 'year']


In our analysis, we will filter the movie "Bluebeard" in the `title` column of the `movie_gross_df` dataframe.

In [ ]:
# filter the Bluebeard movie in movie_gross_df
movie_gross_df[movie_gross_df["title"] == "Bluebeard"]

,title,studio,domestic_gross,foreign_gross,year
317,Bluebeard,Strand,33500.0,5200.0,2010
3045,Bluebeard,WGUSA,43100.0,19400000.0,2017


`movie_gross_df[movie_gross_df["title"] == "Bluebeard"]`: this method filters the rows in the `movie_gross_df` dataframe where the title is equal to **"Bluebeard"**. In order to differentiate these two records, we can add the studio

In [ ]:
# differentiating the Bluebeard movie by adding the studio
movie_gross_df["title"] = movie_gross_df["title"] + " (" + movie_gross_df["studio"].astype(str) + ")"
movie_gross_df["title"].value_counts()[movie_gross_df["title"].value_counts() > 1]

Series([], Name: count, dtype: int64)

### **Data Integrity**
In order to confirm that our dataframes are in the correct format:
- We'll check for missing values in the resulting `movie_df` and `movie_gross_df` dataframes using `.isnull().sum()`
- We'll check for duplicate records in both dataframes using `.duplicated().sum()`
- We'll check for the data types of each column in the Dataframes using `.dtypes`

In [59]:
# function to perform data integrity
def data_integrity():
    # check missing values in movie_df & movie_gross_df
    print(f"Missing values in movie_df\n: {movie_df.isnull().sum()}")
    print(f"Missing values in movie_gross_df:\n {movie_gross_df.isnull().sum()}")

    # check duplicate records in movie_df & movie_gross_df
    print(f"Duplicate records in movie_df: {movie_df.duplicated().sum()}")
    print(f"Duplicate records in movie_gross_df: {movie_gross_df.duplicated().sum()}")

    # check the data types of each column in movie_df & movie_gross_df
    print(f"Column data types in movie_df:\n {movie_df.dtypes}")
    print(f"Column data types in movie_gross_df:\n {movie_gross_df.dtypes}")

# call the data integrity function for movie_df
data_integrity()

Missing values in movie_df
: movie_id           0
primary_title      0
original_title     0
start_year         0
runtime_minutes    0
genres             0
average_rating     0
number_of_votes    0
dtype: int64
Missing values in movie_gross_df:
 title             0
studio            0
domestic_gross    0
foreign_gross     0
year              0
dtype: int64
Duplicate records in movie_df: 0
Duplicate records in movie_gross_df: 0
Column data types in movie_df:
 movie_id            object
primary_title       object
original_title      object
start_year           int64
runtime_minutes    float64
genres              object
average_rating     float64
number_of_votes      int64
dtype: object
Column data types in movie_gross_df:
 title              object
studio             object
domestic_gross    float64
foreign_gross     float64
year                int64
dtype: object


### **Saving the cleaned Dataframes**
Finally, we will save the movie_df and movie_gross_df dataframes to separate CSV files for further EDA analysis.

In [62]:
# save to a csv file
movie_df.to_csv('data/cleaned_data/cleaned_movie_dataset.csv')
movie_gross_df.to_csv('data/cleaned_data/cleaned_movie_gross_dataset.csv')

# HYPOTHESIS TESTING
